# GLiNER distillation

> One GLiNER2 adapter taught by two teachers: decisions from a model fine-tuned to make them, extraction from the model it was fine-tuned from. Weight differences, an adapter started from them, the teachers' answers cached once, rows formatted exactly as each teacher read them, a trainer that pauses and resumes, and the scores that say whether both skills survived.

In [ ]:
#| default_exp distill

In [ ]:
#| export
import copy, hashlib, inspect, json, math, os, random, re, shutil, signal, threading, time, warnings
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import torch
from fastcore.basics import patch
from torch.utils.data import DataLoader

from sysone.checkpoint import TrainingPaused

In [ ]:
#| hide
from fastcore.test import test_eq, test_close, test_fail, test_ne
import logging, tempfile
from sysone.gliner import GlinerDecider
logging.getLogger("transformers").setLevel(logging.ERROR)
warnings.filterwarnings("ignore", message=r".*attn_implementation.*")
warnings.filterwarnings("ignore", message=r".*torch\.jit\.script.*")
FIX = Path("fixtures/tiny-gliner2")

[GLiNER2.5-multi-Decide](https://huggingface.co/fastino/GLiNER2.5-multi-Decide) is [gliner2.5-multi-v1](https://huggingface.co/fastino/gliner2.5-multi-v1) fine-tuned to make decisions: classify a text against labels it reads in the same row. The fine-tuning made it better at that and changed every weight of the encoder, which the extraction heads (entities, relations, JSON records) read too. The aim here is one model with both skills, as an adapter over the base: it should decide like the Decide model and extract like the base.

Two teachers do that without any gold labels. Each row of training text comes with a question, and the teacher that is good at the question answers it once:

- **Decision rows** go to the Decide model. Its answer is a logit for every label, and the student learns those logits, not only the winning label, so it keeps the teacher's confidence.
- **Extraction rows** go to the base. Its answer is the entities, relations and record fields it finds, and the student learns to find the same ones.

The extraction heads stay frozen: they are the base's, and the student's job is to keep feeding them what they expect while its classifier learns to decide. The [tutorial](tutorials/12_system_one_multi.ipynb) runs the whole recipe; this notebook builds its pieces and tests them on two tiny models in `fixtures/tiny-gliner2`, a base and a "tuned" copy changed the way the Decide model differs from its base.

The pieces need sysone's `gliner` extra (`pip install "sysone[gliner]"`).

## Two checkpoints, one architecture

A GLiNER2 checkpoint is a folder: `config.json`, `encoder_config/`, the tokenizer, and every weight in `model.safetensors`. `read_state` reads the weights without building the model, which is all a comparison needs.

In [ ]:
#| export
BASE, DECIDE = "fastino/gliner2.5-multi-v1", "fastino/GLiNER2.5-multi-Decide"

def checkpoint_dir(src, revision:str|None=None) -> Path:
    "A GLiNER2 checkpoint's folder: `src` itself, or its Hub snapshot at `revision` (downloaded once)"
    if Path(str(src)).is_dir(): return Path(src)
    from huggingface_hub import snapshot_download
    return Path(snapshot_download(str(src), revision=revision,
                                  allow_patterns=["*.json", "model.safetensors", "encoder_config/*", "tokenizer*"]))

def hub_revision(src) -> str|None:
    "The commit a Hub model id resolves to now, or None for a local folder"
    if Path(str(src)).is_dir(): return None
    from huggingface_hub import HfApi
    return HfApi().model_info(str(src)).sha

def read_state(src, revision:str|None=None) -> dict:
    "A GLiNER2 checkpoint's weights, as CPU tensors, without building the model"
    from safetensors.torch import load_file
    return load_file(str(checkpoint_dir(src, revision) / "model.safetensors"))

def load_extractor(src, state:dict|None=None, device=None, revision:str|None=None):
    "A GLiNER2 model in eval mode from a folder or the Hub, with `state`'s weights in place of its own when given"
    from gliner2 import AutoExtractor
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")          # DeBERTa-v2 has no sdpa attention: GLiNER2 warns and uses eager
        m = AutoExtractor.from_pretrained(str(checkpoint_dir(src, revision)))
    if state is not None: m.load_state_dict(state)
    return (m.to(device) if device is not None else m).eval()

`weight_delta` compares two checkpoints weight by weight. For each it gives the part of the model it belongs to, how far it moved (the norm of the change over the norm of the weight), the share of its numbers that changed at all, and, when every number of it was multiplied by the same factor, that factor: a weight that was only shrunk had no gradient, just weight decay. `delta_summary` adds the weights up by part.

In [ ]:
#| export
def tensor_part(name:str) -> str:
    "The part of a GLiNER2 boundary model a weight belongs to: a task head, or one of four kinds of encoder weight"
    if not name.startswith("encoder."): return name.split(".")[0]
    if "embeddings" in name and "LayerNorm" not in name: return "encoder embeddings"
    if "LayerNorm" in name: return "encoder norms"
    return "encoder biases" if name.endswith(".bias") else "encoder linears"

def _uniform_scale(w, t, rel):
    "The factor t/w when it is the same for every number of the weight (to 1%), else None"
    if rel == 0 or w.numel() > 2_000_000: return None
    nz = w.abs() > 1e-8
    if not nz.any(): return None
    q = torch.quantile((t[nz] / w[nz]).double(), torch.tensor([0.05, 0.5, 0.95], dtype=torch.double))
    return float(q[1]) if float(q[2] - q[0]) < 0.01 * abs(float(q[1])) else None

def weight_delta(base:dict, tuned:dict) -> list:
    "Per weight: its part, shape, ‖Δ‖/‖W‖, the share of numbers changed, and the factor when the change only rescaled it"
    rows = []
    for k, w in base.items():
        w, t = w.float(), tuned[k].float(); d = t - w
        d2, w2 = float((d * d).sum()), float((w * w).sum())
        rel = math.sqrt(d2 / w2) if w2 else 0.0
        rows.append({"name": k, "part": tensor_part(k), "shape": list(w.shape), "numel": w.numel(), "rel": rel,
                     "changed": float((d != 0).float().mean()), "scale": _uniform_scale(w, t, rel), "d2": d2, "w2": w2})
    return rows

def delta_summary(rows:list) -> list:
    "The change per part: weights, numbers, ‖Δ‖/‖W‖ over the part, the share of numbers changed, and the factor when every weight of the part was only rescaled by it"
    out = []
    for part in dict.fromkeys(r["part"] for r in rows):
        rs = [r for r in rows if r["part"] == part]
        n = sum(r["numel"] for r in rs); scales = [r["scale"] for r in rs]
        d2, w2 = sum(r["d2"] for r in rs), sum(r["w2"] for r in rs)
        out.append({"part": part, "weights": len(rs), "numbers": n, "rel": math.sqrt(d2 / w2) if w2 else (math.inf if d2 else 0.0),
                    "changed": sum(r["changed"] * r["numel"] for r in rs) / n,
                    "scale": float(np.median(scales)) if all(s is not None for s in scales) else None})
    return out

The tiny pair differs as the real one does: every encoder weight moved, the classifier was retrained, the record and relation heads were multiplied by 0.964, and the boundary head (spans) was left alone.

In [ ]:
base_state, tuned_state = read_state(FIX / "base"), read_state(FIX / "tuned")
summary = {r["part"]: r for r in delta_summary(weight_delta(base_state, tuned_state))}
{p: (round(r["rel"], 4), round(r["changed"], 3), r["scale"] and round(r["scale"], 3)) for p, r in summary.items()}

{'boundary_head': (0.0, 0.0, None),
 'classifier': (0.2635, 1.0, None),
 'encoder norms': (0.0103, 1.0, None),
 'encoder embeddings': (0.0085, 1.0, None),
 'encoder biases': (0.0079, 1.0, None),
 'encoder linears': (0.0506, 1.0, None),
 'record_decoder': (0.036, 1.0, 0.964),
 'relation_scorer': (0.036, 1.0, 0.964)}

In [ ]:
#| hide
test_eq(summary["boundary_head"]["changed"], 0.0)
test_close(summary["relation_scorer"]["scale"], 0.964, eps=1e-3)
test_close(summary["record_decoder"]["scale"], 0.964, eps=1e-3)
test_eq(summary["classifier"]["scale"], None)
assert all(summary[p]["changed"] > 0.99 for p in ("encoder linears", "encoder norms", "encoder biases", "encoder embeddings"))

Two models need no training at all. `restore_heads` puts the base's heads back into the tuned model, which costs nothing when those heads were only shrunk. `blend` moves the encoder part of the way from the base to the tuned model, takes the classifier from the tuned model and every other head from the base: at `lam=0` it extracts as the base does, at `lam=1` it decides as the tuned model does, and between the two it trades one for the other.

In [ ]:
#| export
def restore_heads(tuned:dict, base:dict, heads=("relation_scorer", "record_decoder")) -> dict:
    "`tuned`'s weights with the named heads put back to `base`'s"
    return {k: (base[k] if k.split(".")[0] in heads else v) for k, v in tuned.items()}

def blend(base:dict, tuned:dict, lam:float, tuned_heads=("classifier",)) -> dict:
    "The encoder `lam` of the way from `base` to `tuned`, the `tuned_heads` from `tuned`, every other head from `base`"
    out = {}
    for k, w in base.items():
        part = k.split(".")[0]
        if part == "encoder": out[k] = (w.float() + lam * (tuned[k].float() - w.float())).to(w.dtype)
        else: out[k] = tuned[k] if part in tuned_heads else w
    return out

In [ ]:
#| hide
r = restore_heads(tuned_state, base_state)
assert torch.equal(r["relation_scorer.mlp.0.weight"], base_state["relation_scorer.mlp.0.weight"])
assert torch.equal(r["classifier.0.weight"], tuned_state["classifier.0.weight"])
k = "encoder.encoder.layer.0.attention.self.query_proj.weight"
test_close(blend(base_state, tuned_state, 0.5)[k], (base_state[k] + tuned_state[k]) / 2, eps=1e-6)
assert torch.equal(blend(base_state, tuned_state, 1.0)["relation_scorer.mlp.0.weight"], base_state["relation_scorer.mlp.0.weight"])

## An adapter that starts at the tuned model

A LoRA adapter adds `B·A·(alpha/r)` to a linear layer's weight, with `A` of `r` rows. The best rank-`r` approximation of a weight's change `ΔW` is its truncated SVD, `U·S·Vᵀ` cut to the `r` largest singular values, so an adapter can start as close to the tuned model as its rank allows: `B = U·√S` and `A = √S·Vᵀ`, both divided by `√(alpha/r)`. `svd_lora` returns the two factors and the share of `‖ΔW‖²` they keep.

In [ ]:
#| export
def svd_lora(delta:torch.Tensor, r:int, scaling:float=1.0) -> tuple:
    "LoRA factors A (r × in) and B (out × r) whose B·A·scaling is the best rank-r approximation of `delta`, and the share of ‖delta‖² it keeps"
    d = delta.float()
    if min(d.shape) > 4 * r and d.numel() > 50_000_000: U, S, V = torch.svd_lowrank(d, q=r + 16, niter=4); Vh = V.T   # a vocabulary-sized matrix
    else: U, S, Vh = torch.linalg.svd(d, full_matrices=False)
    k = min(r, S.numel())
    root = S[:k].sqrt() / math.sqrt(scaling)
    A, B = torch.zeros(r, d.shape[1]), torch.zeros(d.shape[0], r)
    A[:k], B[:, :k] = root[:, None] * Vh[:k], U[:, :k] * root
    total = float((d * d).sum())
    return A, B, float((S[:k] ** 2).sum()) / total if total else 1.0

In [ ]:
#| hide
g = torch.Generator().manual_seed(0)
low = torch.randn(12, 3, generator=g) @ torch.randn(3, 8, generator=g)
A, B, kept = svd_lora(low, 4, scaling=2.0)
test_eq((A.shape, B.shape), (torch.Size([4, 8]), torch.Size([12, 4])))
test_close(B @ A * 2.0, low, eps=1e-4); test_close(kept, 1.0, eps=1e-5)
A, B, kept = svd_lora(torch.randn(12, 8, generator=g), 2)
assert 0 < kept < 1

`svd_adapter` builds the whole adapter with [PEFT](https://huggingface.co/docs/peft), on the base model:

- every encoder linear (the attention's query, key, value and output, and the feed-forward pair, as GLiNER2's own `encoder` LoRA target picks them) gets LoRA factors from the SVD of its change;
- with `biases=True`, those linears' biases are set to the tuned model's and saved with the adapter (PEFT's `bias="lora_only"`);
- the small modules are carried whole (`modules_to_save`): the classifier, every LayerNorm of the encoder, and its relative-position embeddings, all copied from the tuned model;
- with `embeddings_rank`, the word embeddings' change gets a LoRA of that rank too; without it they stay the base's.

Nothing of the extraction heads is in the adapter: switched on, it changes what they read, never the heads themselves. The base's biases are changed in place, so `disable_adapter()` does not restore them; to compare with the base, load the base again. The report gives, per linear, the share of its change the rank keeps.

In [ ]:
#| export
LORA_PATTERNS = ("query", "key", "value", "dense")      # GLiNER2's `encoder` LoRA target: every linear of a DeBERTa layer

def lora_targets(model) -> list:
    "The encoder's linear layers, by name, as GLiNER2's `encoder` LoRA target resolves them"
    return [n for n, m in model.named_modules() if isinstance(m, torch.nn.Linear) and n.startswith("encoder.")
            and any(p in n.rsplit(".", 1)[-1] for p in LORA_PATTERNS)]

def full_modules(model) -> list:
    "The modules an adapter carries whole: the classifier, every LayerNorm of the encoder, and its relative-position embeddings"
    return ["classifier"] + [n for n, _ in model.named_modules()
                             if n.startswith("encoder.") and (n.endswith("LayerNorm") or n.endswith("rel_embeddings"))]

def _unwrap(model):
    "The GLiNER2 model under a PEFT wrapper"
    return model.get_base_model() if hasattr(model, "get_base_model") else model

def svd_adapter(model, base:dict, tuned:dict, r:int=64, alpha:float|None=None, biases:bool=True,
                embeddings_rank:int|None=None) -> tuple:
    "A PEFT adapter on `model` (loaded with `base`'s weights) that starts as close to `tuned` as rank `r` allows; returns it and a report"
    from peft import LoraConfig, get_peft_model
    alpha = float(alpha or r)
    targets, full = lora_targets(model), full_modules(model)
    emb = "encoder.embeddings.word_embeddings"
    cfg = LoraConfig(r=r, lora_alpha=alpha, target_modules=targets + ([emb] if embeddings_rank else []), modules_to_save=full,
                     bias="lora_only" if biases else "none",
                     rank_pattern={emb: embeddings_rank} if embeddings_rank else {}, alpha_pattern={emb: float(embeddings_rank)} if embeddings_rank else {})
    peft = get_peft_model(model, cfg)
    report = []
    with torch.no_grad():
        for name, mod in _unwrap(peft).named_modules():
            if name in targets:
                key = f"{name}.weight"; s = mod.scaling["default"]
                A, B, kept = svd_lora(tuned[key].float() - base[key].float(), r, s)
                mod.lora_A["default"].weight.copy_(A); mod.lora_B["default"].weight.copy_(B)
                if biases and mod.get_base_layer().bias is not None: mod.get_base_layer().bias.copy_(tuned[f"{name}.bias"])
                report.append({"name": name, "shape": list(tuned[key].shape), "rank": r, "kept": kept})
            elif name == emb and embeddings_rank:
                key = f"{name}.weight"; s = mod.scaling["default"]
                A, B, kept = svd_lora((tuned[key].float() - base[key].float()).T, embeddings_rank, s)   # ΔEᵀ = B·A·s
                mod.lora_embedding_A["default"].copy_(A); mod.lora_embedding_B["default"].copy_(B)
                report.append({"name": name, "shape": list(tuned[key].shape), "rank": embeddings_rank, "kept": kept})
            elif name in full and hasattr(mod, "modules_to_save"):
                mod.modules_to_save["default"].load_state_dict({k[len(name) + 1:]: v for k, v in tuned.items() if k.startswith(name + ".")})
    return peft, report

## What the teachers are asked

A *row* is a text and what to ask about it, written as plain data so it can be stored, split and labelled once:

```python
{"id": "web/0", "kind": "extraction", "source": "web", "lang": "es", "text": "…",
 "ask": {"entities": ["person", "company"], "relations": ["works for"], "structures": {"product": ["name", "price"]},
         "classifications": [{"task": "urgency", "labels": ["0", "1", "2"], "prompt": "How urgent is this?"}]}}
```

`ask_schema` turns the ask into GLiNER2's own `Schema`, built the way GLiNER2's builder builds it. The teacher reads that schema, and the training row the student reads later is built from the same schema with the teacher's answers filled in, so the two rows are the same tokens.

In [ ]:
#| export
def ask_schema(ask:dict, thresholds:dict|None=None):
    "The GLiNER2 `Schema` a row's ask builds: record structures, entity types, relation types, then decisions; a dict instead of a list gives each its description (a record field: its description, or its `field` options); `thresholds` ({'entities': .., 'relations': .., 'structures': ..}) sets each kind's own"
    from gliner2.inference.schema import Schema
    s, th = Schema(), thresholds or {}
    for name, fields in (ask.get("structures") or {}).items():
        b = s.structure(name)
        for f in fields:
            spec = fields[f] if isinstance(fields, dict) else None
            kw = spec if isinstance(spec, dict) else {"description": spec} if spec else {}
            b = b.field(f, **({"threshold": th["structures"]} if th.get("structures") is not None else {}) | kw)
    as_given = lambda x: dict(x) if isinstance(x, dict) else list(x)
    if ask.get("entities"): s = s.entities(as_given(ask["entities"]), **({"threshold": th["entities"]} if th.get("entities") is not None else {}))
    if ask.get("relations"): s = s.relations(as_given(ask["relations"]), **({"threshold": th["relations"]} if th.get("relations") is not None else {}))
    for c in ask.get("classifications") or []:
        labels = {l: c["label_descriptions"].get(l, "") for l in c["labels"]} if c.get("label_descriptions") else list(c["labels"])
        s = s.classification(c["task"], labels, **({"prompt": c["prompt"]} if c.get("prompt") else {}))
    return s

`teacher_answers` runs a model over rows and keeps what it says, in a compact form: the entities, relations and record fields it finds (each with its confidence), and every decision's logits.

- **Extraction** goes through GLiNER2's `batch_extract`, as anyone using the model would, at the model's default threshold.
- **Decisions** need the raw logits, which `batch_extract` turns into probabilities and drops. `decision_logits` reads them where GLiNER2's own classification loss does: it batches the rows with GLiNER2's inference preprocessing, encodes them (`_encode_core`), and runs the classifier on each label's marker. Those are the numbers the student's loss is computed from, on the same tokens.

Both sort the rows by length and batch them under a token budget (`max_tokens`, counted roughly by `row_length`), so each batch holds texts of similar size and a batch of long texts is a small one.

In [ ]:
#| export
def _spans(xs) -> list:
    "[[text, confidence], …] from batch_extract's spans"
    return [[x["text"], round(float(x.get("confidence", 1.0)), 4)] for x in xs or [] if isinstance(x, dict) and x.get("text")]

def _tokens(s:str) -> int:
    "A rough token count: a CJK character is about one token, four characters of other scripts about one"
    cjk = sum(1 for ch in s if "\u2e80" <= ch <= "\u9fff" or "\uac00" <= ch <= "\ud7af" or "\uf900" <= ch <= "\ufaff")
    return cjk + (len(s) - cjk) // 4 + 1

def _ask_parts(ask:dict) -> list:
    "The strings an ask writes into the row: labels, prompts, descriptions, types and fields"
    parts = []
    for c in ask.get("classifications") or []: parts += [c["task"], c.get("prompt") or "", *c["labels"], *(c.get("label_descriptions") or {}).values()]
    for k in ("entities", "relations"):
        x = ask.get(k) or []
        parts += list(x) + ([d for d in x.values() if isinstance(d, str)] if isinstance(x, dict) else [])
    for k, v in (ask.get("structures") or {}).items():
        descs = [d if isinstance(d, str) else (d or {}).get("description") or "" for d in v.values()] if isinstance(v, dict) else []
        parts.append(" ".join([k, *v, *descs]))
    return parts

def row_length(row:dict) -> int:
    "A row's rough length in tokens, its text and what it asks"
    return sum(_tokens(x) + 2 for x in [row["text"]] + _ask_parts(row["ask"]))

def length_batches(lengths:list, bs:int, max_tokens:int|None=None) -> list:
    "Indices sorted by length, cut into batches of up to `bs` items and `max_tokens` padded tokens"
    batches, cur, longest = [], [], 0
    for i in sorted(range(len(lengths)), key=lambda i: lengths[i]):
        n = lengths[i]
        if cur and (len(cur) == bs or (max_tokens and (len(cur) + 1) * max(longest, n) > max_tokens)):
            batches.append(cur); cur, longest = [], 0
        cur.append(i); longest = max(longest, n)
    return batches + ([cur] if cur else [])

def _labels_of(tokens:list) -> list:
    "A classification group's labels, from its schema tokens: the token after each [L]"
    return [tokens[i + 1] for i, t in enumerate(tokens[:-1]) if t == "[L]"]

def decision_logits(model, rows:list, batch_size:int=8, max_tokens:int|None=8192) -> list:
    "Each row's decisions as raw logits, {task: {label: logit}}: the classifier on GLiNER2's encoding of the row, as its classification loss reads it"
    m = _unwrap(model).eval(); proc = m.processor; dev = next(m.parameters()).device
    out = [None] * len(rows)
    with torch.inference_mode():
        for chunk in length_batches([row_length(r) for r in rows], batch_size, max_tokens):
            batch = proc.collate_fn_inference([(rows[i]["text"], ask_schema(rows[i]["ask"]).build()) for i in chunk], architecture="boundary")
            batch = batch.to(dev); core = m._encode_core(batch)
            for k, i in enumerate(chunk):
                tasks, res = [c["task"] for c in rows[i]["ask"].get("classifications") or []], {}
                for cls in core["cls_specs"][k]:
                    toks = batch.schema_tokens_list[k][cls["group_index"]]
                    task = toks[2] if toks[2] in tasks else next((t for t in tasks if toks[2].startswith(t + ":")), None)
                    if task is None: continue
                    states = cls["choice_states"] if "choice_states" in cls else cls["group_embs"][1:]
                    res[task] = {l: round(v, 4) for l, v in zip(_labels_of(toks), m.classifier(states).squeeze(-1).float().tolist())}
                out[i] = res
    return out

def compact_answers(ask:dict, res:dict) -> dict:
    "A batch_extract result, cut to what training needs: {entities, relations, structures}"
    out = {}
    if ask.get("entities"): out["entities"] = {t: _spans((res.get("entities") or {}).get(t)) for t in ask["entities"]}
    if ask.get("relations"):
        found = res.get("relation_extraction") or {}
        out["relations"] = {r: [[p["head"]["text"], p["tail"]["text"], round(min(float(p["head"].get("confidence", 1)), float(p["tail"].get("confidence", 1))), 4)]
                                for p in found.get(r) or [] if isinstance(p, dict) and p.get("head") and p.get("tail")] for r in ask["relations"]}
    if ask.get("structures"):
        out["structures"] = {n: [{f: _spans(inst.get(f)) for f in fields} for inst in res.get(n) or [] if isinstance(inst, dict)]
                             for n, fields in ask["structures"].items()}
    return out

EXTRACTIVE = ("entities", "relations", "structures")

def teacher_answers(model, rows:list, batch_size:int=8, threshold:float=0.5, max_tokens:int|None=8192) -> list:
    "What `model` answers to each row's ask: entities, relations and record fields with confidences (batch_extract), and each decision's logits"
    m = _unwrap(model).eval()
    out = [{} for _ in rows]
    ext = [i for i, r in enumerate(rows) if any(r["ask"].get(k) for k in EXTRACTIVE)]
    with torch.inference_mode():
        for b in length_batches([row_length(rows[i]) for i in ext], batch_size, max_tokens):
            chunk = [ext[k] for k in b]
            res = m.batch_extract([rows[i]["text"] for i in chunk], [ask_schema(rows[i]["ask"]) for i in chunk],
                                  batch_size=len(chunk), threshold=threshold, include_confidence=True)
            for i, r in zip(chunk, res): out[i] = compact_answers(rows[i]["ask"], r)
    dec = [i for i, r in enumerate(rows) if r["ask"].get("classifications")]
    for i, lg in zip(dec, decision_logits(m, [rows[i] for i in dec], batch_size, max_tokens)): out[i]["logits"] = lg
    return out

In [ ]:
rows = [{"id": "r0", "kind": "mixed", "source": "test", "text": "John works at Google and the iPhone costs 999 dollars",
         "ask": {"entities": ["person", "company", "city"], "relations": ["works for", "born in"], "structures": {"product": ["name", "price"]},
                 "classifications": [{"task": "sentiment", "labels": ["positive", "negative"], "prompt": "How does it sound?"}]}},
        {"id": "r1", "kind": "decision", "source": "test", "text": "my card was charged twice",
         "ask": {"classifications": [{"task": "intent", "labels": ["billing", "shipping"]}]}}]
base_model = load_extractor(FIX / "base")
answers = teacher_answers(base_model, rows, threshold=0.2)
answers[0]["entities"], answers[0]["logits"], answers[1]

({'person': [['John', 0.3524],
   ['works', 0.3172],
   ['.', 0.2751],
   ['iPhone', 0.2733],
   ['the', 0.2682],
   ['999', 0.2188]],
  'company': [['John', 0.3524],
   ['works', 0.3172],
   ['.', 0.2752],
   ['iPhone', 0.2733],
   ['the', 0.2682],
   ['999', 0.2188]],
  'city': [['John', 0.3524],
   ['works', 0.3172],
   ['.', 0.2752],
   ['iPhone', 0.2733],
   ['the', 0.2682],
   ['999', 0.2188]]},
 {'sentiment': {'positive': -0.1279, 'negative': -0.1279}},
 {'logits': {'intent': {'billing': -0.1286, 'shipping': -0.1287}}})

At full rank the adapter is the tuned model exactly, everywhere but the word embeddings. Here the tiny encoder's linears are 32 or 64 wide, so rank 32 is full rank for every one of them, and with the embeddings' change at full rank too, the adapted base gives the tuned model's classifier logits (read with `decision_logits`):

In [ ]:
def logits_of(model, text, labels):
    return decision_logits(model, [{"text": text, "ask": {"classifications": [{"task": "label", "labels": labels}]}}])[0]["label"]

text, labels = "the invoice was charged twice and nobody answers", ["billing", "shipping", "other"]
tuned_logits = logits_of(load_extractor(FIX / "tuned"), text, labels)
full, report = svd_adapter(load_extractor(FIX / "base"), base_state, tuned_state, r=32, embeddings_rank=32)
print({l: round(v, 4) for l, v in tuned_logits.items()})
print({l: round(v, 4) for l, v in logits_of(full, text, labels).items()}, "· kept:", round(min(x["kept"] for x in report), 6))

{'billing': -0.3258, 'shipping': -0.3259, 'other': -0.3259}
{'billing': -0.3258, 'shipping': -0.3259, 'other': -0.3259} · kept: 1.0


In [ ]:
#| hide
for l in labels: test_close(logits_of(full, text, labels)[l], tuned_logits[l], eps=1e-3)
low, low_report = svd_adapter(load_extractor(FIX / "base"), base_state, tuned_state, r=2)
assert all(x["kept"] < 1 for x in low_report)
assert max(abs(logits_of(low, text, labels)[l] - tuned_logits[l]) for l in labels) > 1e-4
trainable = {n for n, p in low.named_parameters() if p.requires_grad}
assert any("lora_A" in n for n in trainable) and any("modules_to_save" in n for n in trainable)
assert not any(n.split("base_model.model.")[-1].startswith(("boundary_head", "relation_scorer", "record_decoder")) for n in trainable)
test_eq(len(lora_targets(load_extractor(FIX / "base"))), 2 * 6)

At rank 2 the adapter keeps only part of each change, and the logits move away from the tuned model's. That gap is what training closes.

## Training rows, read as the teacher read them

`training_example` writes a row and its teacher's answers as a GLiNER2 training record. What the teacher found becomes the target; a type it found nothing for stays in the row as a negative. A record structure with no instance keeps its fields empty, which GLiNER2 reads as "none here". GLiNER2 has no way to write a relation type with no pair, so the example lists those types in `absent_relations` and the collator below marks them. Each decision keeps the teacher's logits beside its winning label.

In [ ]:
#| export
def _unique(xs): return list(dict.fromkeys(xs))

def training_example(row:dict, answers:dict|None=None, threshold:float=0.5) -> dict:
    "A GLiNER2 training record from a row's ask and its teacher's answers (`row['teacher']` by default), negatives kept, decisions' logits beside their labels"
    a = answers if answers is not None else row["teacher"]
    sd = copy.deepcopy(ask_schema(row["ask"]).build())
    out, absent = {k: v for k, v in sd.items() if k.endswith("descriptions")}, []
    if sd.get("json_structures"):
        js = []
        for item in sd["json_structures"]:
            (name, fields), = item.items()
            insts = [{f: _unique(t for t, c in inst.get(f, []) if c >= threshold) for f in fields} for inst in (a.get("structures") or {}).get(name, [])]
            insts = [{f: (v if v else "") for f, v in i.items()} for i in insts if any(i.values())]
            js += [{name: i} for i in insts] or [{name: {f: "" for f in fields}}]
        out["json_structures"] = js
    if sd.get("entities"):
        out["entities"] = {t: _unique(x for x, c in (a.get("entities") or {}).get(t, []) if c >= threshold) for t in sd["entities"]}
    if sd.get("relations"):
        rels = []
        for item in sd["relations"]:
            (name, _), = item.items()
            pairs = _unique((h, t) for h, t, c in (a.get("relations") or {}).get(name, []) if c >= threshold and h and t)
            rels += [{name: {"head": h, "tail": t}} for h, t in pairs] or [{name: {"head": "", "tail": ""}}]
            if not pairs: absent.append(name)
        out["relations"] = rels
    if sd.get("classifications"):
        out["classifications"] = []
        for c in sd["classifications"]:
            lg = a["logits"][c["task"]]
            out["classifications"].append({**{k: v for k, v in c.items() if k != "true_label"}, "true_label": [max(c["labels"], key=lambda l: lg[l])],
                                           "teacher_logits": [lg[l] for l in c["labels"]]})
    return {"id": row.get("id"), "kind": row.get("kind"), "input": row["text"], "output": out, "absent_relations": absent}

The collator batches training records through GLiNER2's own preprocessing, in its inference mode: the training mode shuffles labels and fields, swaps synthetic names in and drops some at random, all of which would show the student rows the teacher never saw. It builds the gold targets the same way GLiNER2's evaluation does, then makes two changes before the boundary targets are computed: an absent relation's group gets zero instances, and a decision's target for each label becomes the teacher's probability, `sigmoid(logit / T)`, in place of 1 or 0.

GLiNER2's classifier scores each label with a sigmoid and trains it with binary cross-entropy, so the soft target fits its own loss: the loss is lowest when the student's logit equals the teacher's. At inference, a single-label task's softmax over those logits then gives the teacher's distribution too.

In [ ]:
#| export
class DistillCollator:
    "GLiNER2 batches formatted as at inference, with gold targets, absent relations as negatives and decisions' targets softened to sigmoid(logit / T)"
    def __init__(self, processor, temperature:float=1.0, max_len:int|None=None, max_gold_per_query:int=64):
        self.processor, self.temperature, self.max_len, self.max_gold = processor, temperature, max_len, max_gold_per_query
    def __call__(self, examples:list):
        p = self.processor; p.is_training = False
        kw = {"build_targets": True} if "build_targets" in inspect.signature(p._collate_batch).parameters else {}   # gliner2 ≥ 2.1 asks
        batch = p._collate_batch([(e["input"], copy.deepcopy(e["output"])) for e in examples], max_len=self.max_len, error_policy="raise", **kw)
        for i, e in enumerate(examples):
            absent = set(e.get("absent_relations") or ())
            soft = {c["task"]: c for c in e["output"].get("classifications", []) if "teacher_logits" in c}
            for j, (kind, toks) in enumerate(zip(batch.task_types[i], batch.schema_tokens_list[i])):
                if kind == "relations" and toks[2] in absent: batch.structure_labels[i][j] = [0, []]
                elif kind == "classifications":
                    c = soft.get(toks[2]) or next((c for t, c in soft.items() if toks[2].startswith(t + ":")), None)
                    if c is None: continue
                    lg = dict(zip(c["labels"], c["teacher_logits"]))
                    batch.structure_labels[i][j] = [1 / (1 + math.exp(-lg[l] / self.temperature)) for l in _labels_of(toks)]
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            return p._add_boundary_metadata(batch, "boundary", is_training=False, build_targets=True, max_gold_per_query=self.max_gold,
                                            on_capacity_exceeded="truncate_with_warning")

The check that matters: the training batch holds the very tokens the teacher read, and when the student *is* the teacher (without dropout), the decision loss has nothing to learn: its gradient on the logits is zero.

In [ ]:
examples = [training_example(r, a, threshold=0.2) for r, a in zip(rows, answers)]
proc = base_model.processor
collate = DistillCollator(proc)
batch = collate(examples)
seen = proc.collate_fn_inference([(r["text"], ask_schema(r["ask"]).build()) for r in rows], architecture="boundary")
print("the same tokens as the teacher read:", batch.input_ids.tolist() == seen.input_ids.tolist())
print(examples[0]["output"]["classifications"][0]["true_label"], batch.structure_labels[0][-1])

the same tokens as the teacher read: True
['positive'] [0.468068517160876, 0.468068517160876]


In [ ]:
#| hide
assert batch.input_ids.tolist() == seen.input_ids.tolist()
test_eq(examples[0]["absent_relations"], [r for r in ["works for", "born in"] if not answers[0]["relations"][r]])
grads = {}
def keep(_, __, out): grads["logits"] = out; out.retain_grad()
h = base_model.classifier.register_forward_hook(keep)
base_model.eval()                                                         # no dropout: the logits the teacher gave
with torch.enable_grad(): out = base_model(collate([examples[1]])); out.losses["classification_loss"].backward()
h.remove(); base_model.zero_grad()
test_close(grads["logits"].grad.abs().max().item(), 0.0, eps=1e-4)      # the teacher's own logits (kept to 4 decimals): nothing to learn

## The trainer

`DistillLearner` trains an adapter on training records. It is a plain loop, kept small so that every step does exactly one thing:

- **Batches.** Each epoch shuffles the records with the seed plus the epoch number, cuts them into groups of 50 batches, and sorts each group by length, so a batch holds texts of similar size and wastes little padding. A batch holds up to `bs` records and, with `max_tokens`, no more than that many tokens once padded (an estimate: a character of Chinese, Japanese or Korean counts as a token, four of any other script as one), so a batch of long records is a smaller one: DeBERTa's attention grows with the square of the length, and sixteen rows of a thousand tokens do not fit where sixteen of a hundred do. The order depends only on the seed and the epoch, which is what lets a resumed run see the batches it had not seen yet.
- **Loss.** GLiNER2's own forward pass gives the classification loss (the decisions) and the rest (spans, counts, records, relations). The step minimises `weights["decision"]·decision + weights["extraction"]·extraction`.
- **Parameters.** Only the adapter trains: the LoRA factors at `lr`, the modules carried whole and the biases at `full_lr` (a tenth of `lr` by default). There is no weight decay, which would shrink the carried classifier and norms whether or not they had a gradient. Gradients are clipped at 1.
- **Boundary head settings.** GLiNER2's trainer anneals three of the boundary head's training settings; the head is frozen here, so they stay at the values the base model's training ended with: gold injection at 0.25, the consistency loss at full weight, the soft-IoU term off.
- **Precision.** bf16 autocast on a GPU that supports it, float32 otherwise. GLiNER2's trainer refuses float16 for this architecture by default too.

In [ ]:
#| export
GOLD_INJECTION = 0.25            # GLiNER2's TrainingConfig.gold_injection_end: where the base model's own training ended

def example_length(e:dict) -> int:
    "A training record's rough length in tokens: its text and what it asks, not the answers"
    o = e["output"]
    ask = {"classifications": o.get("classifications"), "entities": list(o.get("entities") or {}),
           "relations": [k for r in o.get("relations", []) for k in r], "structures": {k: list(v) for j in o.get("json_structures", []) for k, v in j.items()}}
    return sum(_tokens(x) + 2 for x in [e["input"]] + _ask_parts(ask))

def epoch_batches(examples:list, bs:int, seed:int, epoch:int, group:int=50, max_tokens:int|None=None, lengths:list|None=None) -> list:
    "One epoch's batches of indices: shuffled by seed and epoch, sorted by length (`lengths`, or measured) within groups of `group` batches, up to `bs` records and `max_tokens` padded tokens each, the batches shuffled again"
    rng = random.Random(f"{seed}:{epoch}")
    if lengths is None: lengths = [example_length(e) for e in examples]
    idx = list(range(len(lengths))); rng.shuffle(idx)
    batches = []
    for s in range(0, len(idx), bs * group):
        chunk = idx[s:s + bs * group]
        batches += [[chunk[k] for k in b] for b in length_batches([lengths[i] for i in chunk], bs, max_tokens)]
    rng.shuffle(batches)
    return batches

In [ ]:
#| hide
fake = [{"input": "x" * n, "output": {}} for n in range(23)]
b0, b1 = epoch_batches(fake, 4, 0, 0, group=2), epoch_batches(fake, 4, 0, 1, group=2)
test_eq(sorted(i for b in b0 for i in b), list(range(23))); test_ne(b0, b1); test_eq(b0, epoch_batches(fake, 4, 0, 0, group=2))
long = [{"input": "x" * (40 * n), "output": {}} for n in range(1, 30)]
bt = epoch_batches(long, 8, 0, 0, max_tokens=600)
assert all(len(b) * max(example_length(long[i]) for i in b) <= 600 or len(b) == 1 for b in bt) and sorted(i for b in bt for i in b) == list(range(29))
test_eq(_tokens("北京大学"), 5); test_eq(_tokens("abcdefgh"), 3)

A run saves a checkpoint every `every` minutes, at each epoch's end, and when it stops early: `wrap_up` minutes before a deadline, or on SIGTERM (what a cloud sends before it stops a machine). The checkpoint holds the trainable weights, the optimizer, the random states and the place in the epoch; it is written the way [sysone's checkpoints](05b_checkpoint.ipynb) are, in `learner-1/` with a `sysone_state.json`, so a [job](40_cloud.ipynb) that stops early ends as *paused*, and `job.resume()` continues it. In a job, `from_env()` reads all of it from the environment the runner sets.

In [ ]:
#| export
class DistillLearner:
    "Train an adapter on training records: decisions softened to one teacher's logits, extraction from the other's findings, the extraction heads frozen"
    def __init__(self, model, train:list, valid:list|None=None, bs:int=16, lr:float=2e-4, full_lr:float|None=None, weights:dict|None=None,
                 temperature:float=1.0, max_len:int|None=None, max_tokens:int|None=4096, warmup:float=0.06, seed:int=0, precision:str="auto",
                 workers:int=0, log_every:int=25, verbose:bool=True, device=None):
        self.model, self.train, self.valid, self.verbose = model, list(train), list(valid or []), verbose
        self.bs, self.lr, self.full_lr = bs, lr, full_lr if full_lr is not None else lr / 10
        self.weights = {"decision": 1.0, "extraction": 1.0} | dict(weights or {})
        self.temperature, self.max_len, self.max_tokens, self.warmup, self.seed = temperature, max_len, max_tokens, warmup, seed
        self.workers, self.log_every = workers, log_every
        self.device = torch.device(device or ("cuda" if torch.cuda.is_available() else "cpu"))
        if precision == "auto": precision = "bf16" if self.device.type == "cuda" and torch.cuda.is_bf16_supported() else "fp32"
        self.precision = precision
        m = _unwrap(model)
        self.collate = DistillCollator(m.processor, temperature, max_len, m.boundary_settings.max_gold_per_query)
        self.log, self.state = [], {"epoch": 0, "batch": 0, "step": 0, "seconds": 0.0, "epochs": None}
        self.ck = self.every = self.deadline = self.resume = None
        self.wrap_up, self.opt, self.sched, self._stop = 10.0, None, None, None
    def __repr__(self): return f"DistillLearner({len(self.train):,} train · {len(self.valid):,} valid · bs {self.bs} · {self.precision} · {self.device})"

    def checkpoints(self, path, every:float|None=30):
        "Save a checkpoint in `path` every `every` minutes, at each epoch's end and when stopping early"
        self.ck, self.every = Path(path), every; return self
    def stop_at(self, deadline:float, wrap_up:float=10):
        "Stop with a checkpoint `wrap_up` minutes before `deadline` (a Unix time)"
        self.deadline, self.wrap_up = float(deadline), float(wrap_up); return self
    def resume_from(self, path):
        "Continue from the checkpoint in `path` when the fit starts"
        self.resume = Path(path); return self
    def from_env(self):
        "Checkpoints, a deadline and a checkpoint to resume from, as a sysone job's runner sets them"
        e = os.environ
        if e.get("SYSONE_CHECKPOINT"): self.checkpoints(Path(e["SYSONE_CHECKPOINT"]) / "learner-1", float(e.get("SYSONE_CHECKPOINT_EVERY") or 0) or None)
        if e.get("SYSONE_DEADLINE"): self.stop_at(float(e["SYSONE_DEADLINE"]), float(e.get("SYSONE_WRAP_UP") or 10))
        if e.get("SYSONE_RESUME"): self.resume_from(Path(e["SYSONE_RESUME"]) / "learner-1")
        return self

    def trainable(self) -> list: return [(n, p) for n, p in self.model.named_parameters() if p.requires_grad]

In [ ]:
#| export
@patch
def _setup(self:DistillLearner, epochs:int):
    "The optimizer and the schedule (warmup, then a cosine decay to zero) for `epochs` epochs"
    self.model.to(self.device)
    lora = [p for n, p in self.trainable() if "lora_" in n]
    full = [p for n, p in self.trainable() if "lora_" not in n]
    self.opt = torch.optim.AdamW([{"params": lora, "lr": self.lr}, {"params": full, "lr": self.full_lr}], weight_decay=0.0)
    self.batches = [epoch_batches(self.train, self.bs, self.seed, e, max_tokens=self.max_tokens) for e in range(epochs)]
    total = sum(len(b) for b in self.batches); warm = max(1, int(self.warmup * total))
    f = lambda s: (s + 1) / warm if s < warm else 0.5 * (1 + math.cos(math.pi * min(1.0, (s - warm) / max(1, total - warm))))
    self.sched = torch.optim.lr_scheduler.LambdaLR(self.opt, f)
    self.state["epochs"], self.state["max_steps"] = epochs, total

@patch
def _loader(self:DistillLearner, examples:list, batches:list):
    "A DataLoader over `batches` of `examples`, collated in `workers` processes (its own generator, so it draws nothing from torch's)"
    return DataLoader([examples[i] for b in batches for i in b], batch_sampler=_Offsets(batches), collate_fn=self.collate,
                      num_workers=self.workers, generator=torch.Generator().manual_seed(self.seed))

class _Offsets:
    "Batches of positions in the flattened list `_loader` hands the DataLoader"
    def __init__(self, batches): self.sizes = [len(b) for b in batches]
    def __len__(self): return len(self.sizes)
    def __iter__(self):
        s = 0
        for n in self.sizes: yield list(range(s, s + n)); s += n

@patch
def _losses(self:DistillLearner, batch) -> tuple:
    "The step's decision and extraction losses, from GLiNER2's own forward"
    batch = batch.to(self.device)
    with torch.autocast(self.device.type, dtype=torch.bfloat16, enabled=self.precision == "bf16"):
        out = self.model(batch)
    dec = (out.losses or {}).get("classification_loss")
    dec = dec.float() if dec is not None else torch.zeros((), device=self.device)
    ext = out.loss.float() - dec if out.loss is not None else torch.zeros((), device=self.device)
    return dec, ext

`fit` trains, `validate` gives the mean losses on the validation records (the model in eval mode), and every logged step goes to `learn.log`. A step's losses come from `_parts`, by name, and `_combine` weighs them into the one it minimises, so a subclass can split the loss differently.

In [ ]:
#| export
@patch
def _parts(self:DistillLearner, batch) -> dict:
    "The step's losses by name: here the decisions and the rest of extraction"
    return dict(zip(("decision", "extraction"), self._losses(batch)))

@patch
def _combine(self:DistillLearner, parts:dict) -> torch.Tensor:
    "The loss a step minimises: the parts, each times its weight"
    return sum(self.weights.get(k, 1.0) * v for k, v in parts.items())

@patch
def validate(self:DistillLearner) -> dict:
    "Mean losses by name over the validation records, the model in eval mode"
    if not self.valid: return {}
    self.model.eval(); sums, n = defaultdict(float), 0
    with torch.no_grad():
        for batch in self._loader(self.valid, epoch_batches(self.valid, self.bs, self.seed, 0, max_tokens=self.max_tokens)):
            k = len(batch); n += k
            for name, v in self._parts(batch).items(): sums[name] += v.item() * k
    self.model.train()
    return {f"valid_{k}": v / n for k, v in sums.items()}

@patch
def fit(self:DistillLearner, epochs:int=2) -> list:
    "Train for `epochs` epochs (continuing a checkpoint when one is set to resume); returns the log"
    self._setup(epochs)
    if self.resume is not None: self._load(self.resume); self.resume = None
    head = _unwrap(self.model).boundary_head
    head.set_gold_injection_prob(GOLD_INJECTION); head.set_consistency_scale(1.0); head.set_soft_iou_scale(0.0)
    params = [p for _, p in self.trainable()]
    t_fit, t_save, step_s, save_s = time.time(), time.time(), 0.0, 0.0
    self._signals(True)
    try:
        for epoch in range(self.state["epoch"], epochs):
            batches = self.batches[epoch]
            start = self.state["batch"]
            self.model.train()
            for k, batch in enumerate(self._loader(self.train, batches[start:]), start):
                t = time.time()
                parts = self._parts(batch)
                loss = self._combine(parts)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(params, 1.0)
                self.opt.step(); self.sched.step(); self.opt.zero_grad(set_to_none=True)
                self.state.update(batch=k + 1, step=self.state["step"] + 1)
                step_s = 0.9 * step_s + 0.1 * (time.time() - t) if step_s else time.time() - t
                if self.state["step"] % self.log_every == 0 or k + 1 == len(batches):
                    self.log.append({"step": self.state["step"], "epoch": epoch + (k + 1) / len(batches), "loss": round(loss.item(), 4)}
                                    | {name: round(v.item(), 4) for name, v in parts.items()}
                                    | {"lr": self.sched.get_last_lr()[0], "seconds": round(self.state["seconds"] + time.time() - t_fit, 1)})
                    if self.verbose: print(json.dumps(self.log[-1]), flush=True)
                reason = self._stop or (self.deadline and time.time() + step_s + save_s > self.deadline - 60 * self.wrap_up and "deadline")
                if reason:
                    self.state["seconds"] += time.time() - t_fit
                    self._save(paused=True, reason=reason)
                    raise TrainingPaused(f"paused ({reason}) at step {self.state['step']}/{self.state['max_steps']}")
                if self.ck and self.every and time.time() - t_save > 60 * self.every:
                    t = time.time(); self._save(); save_s, t_save = time.time() - t, time.time()
            self.state.update(epoch=epoch + 1, batch=0)
            v = self.validate()
            if v:
                self.log.append({"step": self.state["step"], "epoch": epoch + 1} | {k: round(x, 4) for k, x in v.items()})
                if self.verbose: print(json.dumps(self.log[-1]), flush=True)
            if self.ck: self._save()
    finally: self._signals(False)
    self.state["seconds"] += time.time() - t_fit
    if self.ck: self._save(paused=False, reason="complete")
    self.model.eval()
    return self.log

In [ ]:
#| export
@patch
def _signals(self:DistillLearner, on:bool):
    "Stop at the next step on SIGTERM while fitting (main thread only)"
    if threading.current_thread() is not threading.main_thread(): return
    if on:
        self._prev = signal.getsignal(signal.SIGTERM)
        signal.signal(signal.SIGTERM, lambda *_: setattr(self, "_stop", "sigterm"))
    elif getattr(self, "_prev", None) is not None: signal.signal(signal.SIGTERM, self._prev); self._prev = None

@patch
def _save(self:DistillLearner, paused:bool=False, reason:str|None=None):
    "Write the checkpoint: trainable weights, optimizer, random states, and sysone_state.json, replacing the last one whole"
    from safetensors.torch import save_file
    tmp = self.ck.with_name(self.ck.name + ".tmp"); tmp.mkdir(parents=True, exist_ok=True)
    save_file({n: p.detach().float().cpu().contiguous() for n, p in self.trainable()}, str(tmp / "trainable.safetensors"))
    torch.save({"optimizer": self.opt.state_dict(), "scheduler": self.sched.state_dict()}, tmp / "optimizer.pt")
    torch.save({"python": random.getstate(), "numpy": np.random.get_state(), "torch": torch.get_rng_state(),
                "cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None}, tmp / "rng.pt")
    (tmp / "sysone_state.json").write_text(json.dumps(self.state | {"paused": paused, "reason": reason, "fit": 1, "log": self.log}))
    if self.ck.exists():
        old = self.ck.with_name(self.ck.name + ".old"); self.ck.rename(old)
        tmp.rename(self.ck); shutil.rmtree(old)
    else: tmp.rename(self.ck)

@patch
def _load(self:DistillLearner, path):
    "Continue from a checkpoint: weights, optimizer, schedule, random states, the place in the epoch, the log"
    from safetensors.torch import load_file
    path = Path(path); w = load_file(str(path / "trainable.safetensors"))
    params = dict(self.trainable())
    missing = set(params) - set(w)
    if missing: raise ValueError(f"the checkpoint in {path} lacks {len(missing)} trainable weights, e.g. {sorted(missing)[:3]}")
    with torch.no_grad():
        for n, p in params.items(): p.copy_(w[n].to(p.dtype))
    o = torch.load(path / "optimizer.pt", weights_only=False)
    self.opt.load_state_dict(o["optimizer"]); self.sched.load_state_dict(o["scheduler"])
    r = torch.load(path / "rng.pt", weights_only=False)
    random.setstate(r["python"]); np.random.set_state(r["numpy"]); torch.set_rng_state(r["torch"])
    if r.get("cuda") is not None and torch.cuda.is_available(): torch.cuda.set_rng_state_all(r["cuda"])
    s = json.loads((path / "sysone_state.json").read_text())
    self.log = s.pop("log", []); self.state.update({k: s[k] for k in ("epoch", "batch", "step", "seconds")})
    self.state["resumed_from_step"] = s["step"]

A short run on the tiny pair: rows the two tiny teachers answered, an adapter started at rank 2, four epochs. The tiny teacher's labels are close to a coin toss, so the decision loss is near its floor (the targets' own entropy) from the start; what training changes is how far the student's logits are from the teacher's, and the extraction loss:

In [ ]:
tuned_model = load_extractor(FIX / "tuned")
texts = ["John works at Google in London", "Mary sold her car to Peter for 300 dollars", "the invoice was charged twice",
         "Acme Corp opened an office in Paris last year", "my parcel never arrived and I want a refund", "Lisa joined IBM as an engineer"]
dec_rows = [{"id": f"d{i}", "kind": "decision", "source": "test", "text": t,
             "ask": {"classifications": [{"task": "intent", "labels": ["billing", "shipping", "work", "sales"]}]}} for i, t in enumerate(texts)]
ext_rows = [{"id": f"x{i}", "kind": "extraction", "source": "test", "text": t,
             "ask": {"entities": ["person", "company", "city"], "relations": ["works for"]}} for i, t in enumerate(texts)]
dec_answers = teacher_answers(tuned_model, dec_rows)
train_ex = [training_example(r, a, threshold=0.2) for r, a in zip(dec_rows, dec_answers)] + \
           [training_example(r, a, threshold=0.2) for r, a in zip(ext_rows, teacher_answers(base_model, ext_rows, threshold=0.2))]
def logit_gap(model):
    "The mean distance between the model's decision logits and the teacher's"
    got = decision_logits(model, dec_rows)
    return float(np.mean([abs(g[t][l] - a["logits"][t][l]) for g, a in zip(got, dec_answers) for t in a["logits"] for l in a["logits"][t]]))
def tiny_learner(**kw):
    torch.manual_seed(0)
    m, _ = svd_adapter(load_extractor(FIX / "base"), base_state, tuned_state, r=2)
    return DistillLearner(m, train_ex, train_ex[::3], bs=4, lr=5e-3, log_every=1, verbose=False, **kw)
learn = tiny_learner()
gap_before = logit_gap(learn.model)
log = learn.fit(4)
print(f"distance to the teacher's logits: {gap_before:.4f} → {logit_gap(learn.model):.4f}")
[(x["step"], x.get("valid_decision"), x.get("valid_extraction")) for x in log if "valid_decision" in x]

distance to the teacher's logits: 0.0634 → 0.0134


[(3, 0.68, 2.7855), (6, 0.68, 2.7731), (9, 0.68, 2.7668), (12, 0.68, 2.7655)]

In [ ]:
#| hide
v = [x for x in log if "valid_decision" in x]
assert logit_gap(learn.model) < 0.8 * gap_before and v[-1]["valid_extraction"] < v[0]["valid_extraction"]
test_eq(learn.state["step"], 4 * 3)

A paused run continues where it stopped and ends with the weights an uninterrupted run gets. Here the deadline falls inside the second epoch:

In [ ]:
ck = Path(tempfile.mkdtemp()) / "learner-1"
whole = tiny_learner(); whole.fit(2)
first = tiny_learner().checkpoints(ck, every=None)
steps = {"n": 0}
orig = first._losses
def counting(batch):
    if first.model.training:                                   # training steps, not validation
        steps["n"] += 1
        if steps["n"] == 5: first._stop = "sigterm"          # as if the machine were being stopped
    return orig(batch)
first._losses = counting
test_fail(lambda: first.fit(2), contains="paused (sigterm) at step 5/6")
state = json.loads((ck / "sysone_state.json").read_text())
second = tiny_learner().resume_from(ck); second.fit(2)
print(state["paused"], state["reason"], state["step"], "→", second.state["step"])
same = all(torch.allclose(a, b, atol=1e-5) for (_, a), (_, b) in zip(whole.trainable(), second.trainable()))
print("resumed weights equal the uninterrupted run's:", same)

True sigterm 5 → 6
resumed weights equal the uninterrupted run's: True


In [ ]:
#| hide
assert same and state["paused"] and state["epoch"] == 1 and state["batch"] == 2
test_eq(second.state["resumed_from_step"], 5)

## Scores

A candidate model is scored on rows by asking it what the teachers were asked (`teacher_answers` again) and comparing:

- with **gold** labels, where a row has them: decision accuracy, its calibration (ECE: how far the confidence is from the accuracy, over 10 bins), and entity F1, a predicted entity counting as right when its type and text (case and spacing aside) match a gold one;
- with a **reference**, the teacher's own answers on held-out rows: how often the top label agrees, the mean total-variation distance between the two label distributions (0 when they are equal, 1 when they share nothing), and F1 against the reference's entities, relation triples and record fields.

In [ ]:
#| export
def _norm(s) -> str: return re.sub(r"\s+", " ", str(s)).strip().lower()

def _softmax(xs) -> np.ndarray:
    x = np.asarray(xs, float); e = np.exp(x - x.max()); return e / e.sum()

def _f1(tp, fp, fn) -> float: return 2 * tp / (2 * tp + fp + fn) if tp + fp + fn else float("nan")

def _sets(ans:dict) -> dict:
    "An answer's entities, relation triples and record fields (all it returned, at the threshold it was asked with) as sets of normalised tuples"
    return {"entities": {(t, _norm(x)) for t, xs in (ans.get("entities") or {}).items() for x, _ in xs},
            "relations": {(r, _norm(h), _norm(t)) for r, ps in (ans.get("relations") or {}).items() for h, t, _ in ps},
            "records": {(n, f, _norm(x)) for n, insts in (ans.get("structures") or {}).items() for i in insts for f, xs in i.items() for x, _ in xs}}

def ece(conf, right, bins:int=10) -> float:
    "Expected calibration error: the gap between confidence and accuracy, averaged over `bins` equal-width confidence bins"
    conf, right = np.asarray(conf, float), np.asarray(right, float)
    if not len(conf): return float("nan")
    b = np.minimum((conf * bins).astype(int), bins - 1)
    return float(sum(abs(conf[b == k].mean() - right[b == k].mean()) * (b == k).mean() for k in range(bins) if (b == k).any()))

def scores(pred:list, rows:list, ref:list|None=None) -> dict:
    "Decision accuracy, ECE and entity F1 against the rows' gold; top-label agreement, total-variation distance and entity, relation and record F1 against `ref`"
    acc, conf, agree, tv = [], [], [], []
    cnt = {f"{k}_{w}": [0, 0, 0] for k in ("entities", "relations", "records") for w in ("gold", "ref")}
    for i, (p, row) in enumerate(zip(pred, rows)):
        gold = row.get("gold") or {}
        for task, lg in (p.get("logits") or {}).items():
            labels = list(lg); pp = _softmax([lg[l] for l in labels]); top = labels[int(pp.argmax())]
            g = (gold.get("classifications") or {}).get(task)
            if g is not None: acc.append(top == g); conf.append(float(pp.max()))
            if ref is not None and task in (ref[i].get("logits") or {}):
                r = ref[i]["logits"][task]; pr = _softmax([r[l] for l in labels])
                agree.append(labels[int(pr.argmax())] == top); tv.append(0.5 * float(np.abs(pp - pr).sum()))
        ps = _sets(p)
        sources = ([("gold", {"entities": {(t, _norm(x)) for t, xs in gold["entities"].items() for x in xs}})] if gold.get("entities") is not None else []) + \
                  ([("ref", _sets(ref[i]))] if ref is not None else [])
        for w, s in sources:
            for k in ("entities", "relations", "records"):
                if k not in s or not (row["ask"].get(k) or (k == "records" and row["ask"].get("structures"))): continue
                c = cnt[f"{k}_{w}"]; c[0] += len(ps[k] & s[k]); c[1] += len(ps[k] - s[k]); c[2] += len(s[k] - ps[k])
    out = {}
    if acc: out |= {"decisions": len(acc), "accuracy": float(np.mean(acc)), "ece": ece(conf, acc)}
    if agree: out |= {"agreement": float(np.mean(agree)), "tv": float(np.mean(tv)), "decisions_ref": len(agree)}
    for k, (tp, fp, fn) in cnt.items():
        if tp + fp + fn: out[f"{k.split('_')[0]}_f1_{k.split('_')[1]}"] = _f1(tp, fp, fn)
    return out

def suite_scores(pred:list, rows:list, ref:list|None=None, by:str="suite") -> dict:
    "`scores` per suite (the rows' `by` field)"
    groups = defaultdict(list)
    for i, r in enumerate(rows): groups[r.get(by) or r.get("source")].append(i)
    return {g: scores([pred[i] for i in ix], [rows[i] for i in ix], [ref[i] for i in ix] if ref is not None else None) for g, ix in groups.items()}

In [ ]:
eval_rows = [{"text": t, "suite": "tiny", "ask": {"entities": ["person", "company"], "classifications": [{"task": "intent", "labels": ["billing", "work"]}]},
              "gold": {"classifications": {"intent": "work" if "work" in t or "joined" in t else "billing"}, "entities": {"person": [], "company": []}}}
             for t in texts]
ref = teacher_answers(tuned_model, eval_rows, threshold=0.2)
suite_scores(teacher_answers(learn.model, eval_rows, threshold=0.2), eval_rows, ref)

{'tiny': {'decisions': 6,
  'accuracy': 0.8333333333333334,
  'ece': 0.33332916666667023,
  'agreement': 0.8333333333333334,
  'tv': 4.166666663192135e-06,
  'decisions_ref': 6,
  'entities_f1_gold': 0.0,
  'entities_f1_ref': 0.8823529411764706}}

In [ ]:
#| hide
s = scores(ref, eval_rows, ref)
test_eq((s["agreement"], s["tv"], s["entities_f1_ref"]), (1.0, 0.0, 1.0))
test_close(ece([0.9, 0.9, 0.6, 0.6], [1, 1, 0, 1]), 0.5 * 0.1 + 0.5 * 0.1)
s = scores([{"entities": {"person": [["John", 0.9], ["Mary", 0.8]]}}], [{"ask": {"entities": ["person"]}, "gold": {"entities": {"person": ["john", "Peter"]}}}])
test_close(s["entities_f1_gold"], 0.5)

## The export and its card

`export_adapter` writes the adapter in PEFT's format, which loads over the base with `PeftModel.from_pretrained`, beside a `sysone_gliner.json` that names the base (so `sysone.load_learner` and `GlinerDecider.load` load it as a GLiNER2 model with its adapter on) and a `training.json` with the run's record. `adapter_card` writes the model card from them, and `push_adapter` uploads the folder to a Hub model repo, private by default.

In [ ]:
#| export
def distill_record(learn:DistillLearner, **extra) -> dict:
    "What trained an adapter: the code, the environment and hardware, the run's settings, the time it took and its log; plus `extra` (teachers, data, adapter, scores)"
    from sysone.record import environment, hardware, git_info, entry_point
    import sysone, datetime
    shipped = json.loads(os.environ.get("SYSONE_SOURCE") or "{}")
    kinds = Counter(e.get("kind") for e in learn.train)
    return {"recorded": datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"),
            "code": {"git": git_info() or shipped.get("git"), "sysone": sysone.__version__, "entry": entry_point()},
            "environment": environment(), "hardware": hardware(), "job": json.loads(os.environ.get("SYSONE_JOB") or "null"),
            "training": {"epochs": learn.state.get("epochs"), "steps": learn.state["step"], "bs": learn.bs, "lr": learn.lr, "full_lr": learn.full_lr,
                         "weights": learn.weights, "temperature": learn.temperature, "precision": learn.precision, "seed": learn.seed, "max_tokens": learn.max_tokens,
                         "seconds": round(learn.state["seconds"], 1), "train": len(learn.train), "valid": len(learn.valid), "kinds": dict(kinds),
                         "trainable": sum(p.numel() for _, p in learn.trainable()), "resumed_from_step": learn.state.get("resumed_from_step"),
                         "log": learn.log}} | extra

def export_adapter(model, path, base:str, record:dict|None=None, name:str|None=None) -> Path:
    "The adapter in PEFT's format, `sysone_gliner.json` naming its base, and the training record"
    path = Path(path); path.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(str(path))
    (path / "sysone_gliner.json").write_text(json.dumps({"backend": "gliner2", "base": base, "adapter": True, "regime": "two-teacher distillation",
                                                          "temperature": {}, "name": name or path.name}, indent=2))
    if record is not None: (path / "training.json").write_text(json.dumps(record, indent=1, default=str))
    return path

In [ ]:
#| hide
out = export_adapter(learn.model, Path(tempfile.mkdtemp()) / "tiny-adapter", base=str(FIX / "base"), record=distill_record(learn, teachers={"test": None}))
test_eq(sorted(p.name for p in out.iterdir()), ["README.md", "adapter_config.json", "adapter_model.safetensors", "sysone_gliner.json", "training.json"])
gd = GlinerDecider.load(out)
again = teacher_answers(gd.model, eval_rows, threshold=0.2)
trained = teacher_answers(learn.model, eval_rows, threshold=0.2)
for a, b in zip(again, trained):
    for l in a["logits"]["intent"]: test_close(a["logits"]["intent"][l], b["logits"]["intent"][l], eps=1e-3)

Loaded again from the export, over a fresh base, the adapter answers as the trained model does. The card says what the adapter is, how to load it with `gliner2` and `peft` alone, what it scored next to the two teachers, and what trained it.

In [ ]:
#| export
def _table(rows:list, cols:list) -> str:
    fmt = lambda v: "–" if v is None or (isinstance(v, float) and math.isnan(v)) else (f"{v:.3f}" if isinstance(v, float) else str(v))
    return "\n".join(["| " + " | ".join(c for c, _ in cols) + " |", "|" + "---|" * len(cols)] +
                     ["| " + " | ".join(fmt(r.get(k)) for _, k in cols) + " |" for r in rows])

def adapter_card(path, repo_id:str, base:str=BASE, teachers:dict|None=None, description:str|None=None, license:str="apache-2.0",
                 scores:dict|None=None, limitations:list|None=None) -> str:
    "A model card for an adapter export: what it is, how to load it, its `scores` ({model: {column: value}}) beside the teachers', its limitations, data and training"
    path = Path(path)
    rec = json.loads((path / "training.json").read_text()) if (path / "training.json").exists() else {}
    tr, data = rec.get("training") or {}, rec.get("data") or {}
    cols = list(dict.fromkeys(c for v in (scores or {}).values() for c in v))
    sc = {"table": _table([{"model": m, **v} for m, v in scores.items()], [("model", "model")] + [(c, c) for c in cols])} if scores else rec.get("scores") or {}
    teachers = teachers or rec.get("teachers") or {}          # "repo@revision" in a training record
    repo = lambda t: t.split("@")[0]
    front = ["---", f"license: {license}", f"base_model: {base}", "library_name: peft", "pipeline_tag: schema-extraction",
             "tags:", *[f"- {t}" for t in ("gliner2", "lora", "distillation", "decision-model", "named-entity-recognition", "relation-extraction", "sysone")],
             "language:", "- multilingual", "---", ""]
    body = [f"# {repo_id.split('/')[-1]}", "",
            description or f"A LoRA adapter for [{base}](https://huggingface.co/{base}) that makes its decisions like "
                           f"[{repo(teachers.get('decision', DECIDE))}](https://huggingface.co/{repo(teachers.get('decision', DECIDE))}) and keeps the base's own extraction "
                           "(entities and JSON records). It was distilled from both models with [sysonelib](https://github.com/sgaseretto/sysonelib).", "",
            "## Use it", "", "```python", "from gliner2 import AutoExtractor", "from peft import PeftModel", "",
            f'model = PeftModel.from_pretrained(AutoExtractor.from_pretrained("{base}"), "{repo_id}").get_base_model()',
            'model.classify_text("My card was charged twice", {"intent": ["billing", "shipping", "account"]})',
            'model.extract_entities("Ana Ruiz joined Acme in Madrid", ["person", "company", "city"])', "```", "",
            "The adapter is not merged: the base's weights stay as they are, and the adapter adds low-rank changes to the encoder's linear "
            "layers, plus its own copies of the classifier, the encoder's LayerNorms and its relative-position embeddings, and new biases for the adapted "
            "linears. The span, relation and record heads are the base's, untouched.", ""]
    if sc:
        body += ["## Scores", "", "Decisions are scored against gold labels and against the decision teacher's answers; extraction against gold where a "
                 "suite has it and against the extraction teacher's answers on held-out text. Agreement is how often the top label is the teacher's; "
                 "ECE is the calibration error (lower is better).", "", sc.get("table", ""), ""]
    if limitations: body += ["## Limitations", ""] + [f"- {x}" for x in limitations] + [""]
    if data:
        body += ["## Data", "", "No gold labels were used for training: every row's text came from the sources below, and the teachers answered it.", "",
                 _table([{"source": k, **v} for k, v in (data.get("sources") or {}).items()],
                        [("source", "source"), ("rows used (training and evaluation)", "rows"), ("licence", "license"), ("use", "use")]), ""]
    if tr:
        body += ["## Training", "", f"- Teachers: {', '.join(f'{k}: `{v}`' for k, v in teachers.items())}",
                 f"- {tr.get('train', 0):,} training rows ({', '.join(f'{k} {v:,}' for k, v in (tr.get('kinds') or {}).items())}), {tr.get('epochs')} epochs, "
                 f"{tr.get('steps', 0):,} steps of {tr.get('bs')}, learning rates {tr.get('lr')} (LoRA) and {tr.get('full_lr')} (whole modules), {tr.get('precision')}",
                 f"- Loss: the classifier's binary cross-entropy against the decision teacher's sigmoid(logit / {tr.get('temperature')}), plus GLiNER2's extraction "
                 f"losses against the extraction teacher's findings, weighted {tr.get('weights')}",
                 f"- {tr.get('trainable', 0):,} trainable numbers; {tr.get('seconds', 0) / 60:.0f} minutes on {', '.join(((rec.get('hardware') or {}).get('accelerator') or {}).get('devices') or ['?'])}"]
        if rec.get("adapter"): body.append(f"- Adapter: {rec['adapter'].get('summary', '')}")
        body.append("")
    (path / "README.md").write_text("\n".join(front + body))
    return "\n".join(front + body)

def push_adapter(path, repo_id:str, private:bool=True, commit_message:str|None=None, token=None, **card_kw) -> dict:
    "Write the card and upload an adapter export to a Hub model repo, private by default"
    from huggingface_hub import HfApi
    adapter_card(path, repo_id, **card_kw)
    api = HfApi(token=token)
    api.create_repo(repo_id, private=private, exist_ok=True)
    info = api.upload_folder(repo_id=repo_id, folder_path=str(path), commit_message=commit_message or "Adapter, card and training record")
    return {"repo": repo_id, "private": private, "commit": info.oid, "url": info.commit_url}

In [ ]:
#| hide
card = adapter_card(out, "someone/tiny-adapter", base=str(FIX / "base"), scores={"base": {"accuracy": 0.5, "F1": 0.3}, "adapter": {"accuracy": 0.6, "F1": 0.31}},
                    teachers={"decision": "a/decide@123", "extraction": "a/base@456"}, limitations=["Relations are not scored."])
assert "## Limitations\n\n- Relations are not scored." in card and "(https://huggingface.co/a/decide)" in card and "`a/decide@123`" in card
assert "PeftModel.from_pretrained" in card and "## Training" in card and (out / "README.md").exists()
assert "| model | accuracy | F1 |" in card and "| adapter | 0.600 | 0.310 |" in card

## Rows from public datasets

Training needs texts and questions, not answers: the teachers answer. A labelled dataset still helps twice: its labels say which entity types or intents fit its texts, so the questions are sensible, and its test split, with the gold kept, becomes an evaluation suite. These converters turn rows of a few common formats into rows with an `ask`.

Entity datasets tag tokens with BIO tags: `B-type` starts a mention, `I-type` continues it, `O` is outside every mention. `bio_mentions` reads the mentions back, joining tokens with a space (or with nothing, for languages written without spaces).

In [ ]:
#| export
def bio_mentions(tokens:list, tags:list, join:str=" ") -> dict:
    "Mentions by type from BIO tags ('B-type', 'I-type', 'O'): {type: [mention, …]}"
    out, cur, typ = defaultdict(list), [], None
    for tok, tag in list(zip(tokens, tags)) + [(None, "O")]:
        inside = tag.startswith("I-") and tag[2:] == typ
        if cur and not inside: out[typ].append(join.join(cur)); cur, typ = [], None
        if tag.startswith(("B-", "I-")) and not inside: cur, typ = [tok], tag[2:]
        elif inside: cur.append(tok)
    return dict(out)

In [ ]:
#| hide
test_eq(bio_mentions(["Ana", "Ruiz", "joined", "Acme", "Corp", "in", "Madrid"], ["B-PER", "I-PER", "O", "B-ORG", "I-ORG", "O", "B-LOC"]),
        {"PER": ["Ana Ruiz"], "ORG": ["Acme Corp"], "LOC": ["Madrid"]})
test_eq(bio_mentions(list("北京大学"), ["B-ORG", "I-ORG", "I-ORG", "I-ORG"], join=""), {"ORG": ["北京大学"]})
test_eq(bio_mentions(["a", "b"], ["I-X", "I-Y"]), {"X": ["a"], "Y": ["b"]})

The tags' names are codes (`HumanSettlement`, `programlang`); GLiNER reads type names as words, so each dataset's codes get plain names. A row asks for the types its text has plus some it lacks, about a third, so the model also learns to find nothing.

In [ ]:
#| export
MULTICONER_TYPES = {"Facility": "facility", "OtherLOC": "location", "HumanSettlement": "city or town", "Station": "station",
                    "VisualWork": "film or TV show", "MusicalWork": "song or album", "WrittenWork": "book or written work", "ArtWork": "artwork",
                    "Software": "software", "OtherCW": "creative work", "MusicalGRP": "band", "PublicCorp": "public company",
                    "PrivateCorp": "private company", "OtherCorp": "company", "AerospaceManufacturer": "aerospace manufacturer",
                    "SportsGRP": "sports team", "CarManufacturer": "car manufacturer", "TechCORP": "tech company", "ORG": "organization",
                    "Scientist": "scientist", "Artist": "artist", "Athlete": "athlete", "Politician": "politician", "Cleric": "cleric",
                    "SportsManager": "sports manager", "OtherPER": "person", "Clothing": "clothing", "Vehicle": "vehicle", "Food": "food",
                    "Drink": "drink", "OtherPROD": "product", "Medication/Vaccine": "medication or vaccine", "MedicalProcedure": "medical procedure",
                    "AnatomicalStructure": "anatomical structure", "Symptom": "symptom", "Disease": "disease"}
CROSSNER_TYPES = {"academicjournal": "academic journal", "astronomicalobject": "astronomical object", "chemicalcompound": "chemical compound",
                  "chemicalelement": "chemical element", "literarygenre": "literary genre", "musicalartist": "musical artist",
                  "musicalinstrument": "musical instrument", "musicgenre": "music genre", "politicalparty": "political party",
                  "programlang": "programming language", "misc": "other", "organisation": "organization"}

def sample_types(present:list, pool:list, rng:random.Random, absent:float=1/3, most:int=8) -> list:
    "The types a row asks for: those its text has (up to `most`), and about `absent` of the row's types from `pool` that it lacks, shuffled"
    present = _unique(present)[:most]
    others = [t for t in pool if t not in present]
    k = min(len(others), max(1, round(len(present) * absent / (1 - absent)))) if present else min(len(others), rng.randint(2, 4))
    out = present + rng.sample(others, k); rng.shuffle(out)
    return out

def entity_row(id:str, text:str, mentions:dict, types:list, source:str, lang:str|None=None, gold:bool=False, suite:str|None=None) -> dict:
    "A row asking for entity `types` in `text`; with `gold`, the mentions are kept as its gold"
    row = {"id": id, "kind": "extraction", "source": source, "lang": lang, "text": text, "ask": {"entities": list(types)}}
    if gold: row["gold"] = {"entities": {t: list(mentions.get(t, [])) for t in types}}
    if suite: row["suite"] = suite
    return row

def tagged_rows(ds, names:dict|None, source:str, lang:str|None=None, n:int|None=None, seed:int=0, gold:bool=False, all_types:bool=False,
                join:str=" ", suite:str|None=None, tag_names:list|None=None, types:list|None=None) -> list:
    "Rows from a BIO-tagged dataset (`tokens`, `ner_tags` as names, or as indices into `tag_names` or the dataset's own); type codes renamed by `names`, the types asked drawn from `types` (all the tags' types by default)"
    rng = random.Random(seed)
    feats = getattr(ds, "features", None)
    if tag_names is None and feats is not None and hasattr(feats.get("ner_tags"), "feature") and hasattr(feats["ner_tags"].feature, "names"):
        tag_names = feats["ner_tags"].feature.names
    rows = list(ds)
    if n is not None and n < len(rows): rows = rng.sample(rows, n)
    tags_of = lambda r: list(r["ner_tags"]) if not len(r["ner_tags"]) or isinstance(r["ner_tags"][0], str) else [tag_names[k] for k in r["ner_tags"]]
    name = lambda t: (names or {}).get(t, t)
    codes = [t[2:] for t in tag_names if t != "O"] if tag_names else sorted({t[2:] for r in rows for t in tags_of(r) if t != "O"})
    pool = list(types) if types is not None else _unique(name(c) for c in codes)
    out = []
    for i, r in enumerate(rows):
        tags = tags_of(r)
        found = {}
        for c, ms in bio_mentions(r["tokens"], tags, join).items():
            if name(c) in pool: found.setdefault(name(c), []).extend(ms)
        types = pool if all_types else sample_types(list(found), pool, rng)
        out.append(entity_row(f"{source}/{r.get('id', i)}", join.join(r["tokens"]), found, types, source, lang, gold, suite))
    return out

In [ ]:
fake = [{"id": "0", "tokens": ["Ana", "flew", "to", "Lima", "with", "Avianca"], "ner_tags": ["B-OtherPER", "O", "O", "B-HumanSettlement", "O", "B-PrivateCorp"]},
        {"id": "1", "tokens": ["Aspirin", "eases", "headaches"], "ner_tags": ["B-Medication/Vaccine", "O", "B-Symptom"]}]
tagged_rows(fake, MULTICONER_TYPES, "multiconer", "es", gold=True)[0]

{'id': 'multiconer/0',
 'kind': 'extraction',
 'source': 'multiconer',
 'lang': 'es',
 'text': 'Ana flew to Lima with Avianca',
 'ask': {'entities': ['private company', 'city or town', 'person', 'symptom']},
 'gold': {'entities': {'private company': ['Avianca'],
   'city or town': ['Lima'],
   'person': ['Ana'],
   'symptom': []}}}

In [ ]:
#| hide
r = tagged_rows(fake, MULTICONER_TYPES, "multiconer", "es", gold=True)
test_eq(r[0]["gold"]["entities"]["city or town"], ["Lima"]); assert {"person", "city or town", "private company"} <= set(r[0]["ask"]["entities"])
assert len(r[0]["ask"]["entities"]) > 3          # and some types the text lacks
test_eq(set(tagged_rows(fake, MULTICONER_TYPES, "m", all_types=True)[1]["ask"]["entities"]), {"person", "city or town", "private company", "medication or vaccine", "symptom"})
test_eq(tagged_rows(fake, MULTICONER_TYPES, "m", gold=True, all_types=True, types=["symptom"])[1]["gold"], {"entities": {"symptom": ["headaches"]}})

[NuNER](https://huggingface.co/datasets/numind/NuNER) annotates web text with free-form entity types, written as a string such as `"['Ana Ruiz <> person', 'Acme <> company']"`. Its types are many and varied, which is the point: the student should handle names it has not seen.

In [ ]:
#| export
def nuner_mentions(output:str) -> dict:
    "NuNER's annotation string, \"['mention <> type', …]\", as {type: [mention, …]}"
    import ast
    try: items = ast.literal_eval(output)
    except (ValueError, SyntaxError): return {}
    out = defaultdict(list)
    for it in items:
        if isinstance(it, str) and " <> " in it:
            m, t = it.rsplit(" <> ", 1); out[t.strip().lower()].append(m.strip())
    return dict(out)

def nuner_rows(ds, source:str="nuner", n:int|None=None, seed:int=0) -> list:
    "Rows from NuNER (`input`, `output`): the text's own types plus types other texts have"
    rng = random.Random(seed); rows = list(ds)
    if n is not None and n < len(rows): rows = rng.sample(rows, n)
    found = [nuner_mentions(r["output"]) for r in rows]
    pool = _unique(t for f in found for t in f)
    return [entity_row(f"{source}/{i}", r["input"], f, sample_types(list(f), pool, rng), source, "en") for i, (r, f) in enumerate(zip(rows, found)) if f]

In [ ]:
#| hide
test_eq(nuner_mentions("['State University of New York Press <> Publisher', 'NY <> place']"), {"publisher": ["State University of New York Press"], "place": ["NY"]})
test_eq(nuner_mentions("not a list"), {})

Decisions come from datasets labelled with intents, topics or sentiment. `choice_row` asks one question with a set of labels. For training, a row offers its gold label and up to `k - 1` others drawn at random, so the same text comes with different choices, and the teacher's logits say how it ranks all of them; for evaluation, a row offers every label and keeps the gold.

In [ ]:
#| export
def _human(s) -> str: return str(s).replace("_", " ").strip()

def choice_row(id:str, text:str, labels:list, gold:str|None, task:str, source:str, lang:str|None=None, prompt:str|None=None,
               rng:random.Random|None=None, k:int|None=None, keep_gold:bool=False, suite:str|None=None) -> dict:
    "A row asking one single-label question: all `labels`, or the gold and k-1 others; with `keep_gold`, the gold label is kept for scoring"
    if k is not None and len(labels) > k:
        rest = [l for l in labels if l != gold]
        labels = ([gold] if gold is not None else []) + (rng or random).sample(rest, k - (gold is not None))
        (rng or random).shuffle(labels)
    c = {"task": task, "labels": list(labels)} | ({"prompt": prompt} if prompt else {})
    row = {"id": id, "kind": "decision", "source": source, "lang": lang, "text": text, "ask": {"classifications": [c]}}
    if keep_gold and gold is not None: row["gold"] = {"classifications": {task: gold}}
    if suite: row["suite"] = suite
    return row

def labelled_rows(ds, text:str, label:str, task:str, source:str, prompt:str|None=None, names:list|None=None, lang=None, n:int|None=None,
                  seed:int=0, k:int|None=12, keep_gold:bool=False, suite:str|None=None) -> list:
    "Choice rows from a labelled dataset; labels are class indices into `names` or strings, '_' read as spaces; `lang` a column name or a code"
    rng = random.Random(seed); rows = list(ds)
    if n is not None and n < len(rows): rows = rng.sample(rows, n)
    to_name = lambda v: _human(names[v] if names is not None and isinstance(v, int) else v)
    labels = _unique(to_name(v) for v in names) if names is not None else _unique(to_name(r[label]) for r in rows)
    get_lang = (lambda r: r.get(lang)) if isinstance(lang, str) and rows and lang in rows[0] else (lambda r: lang)
    return [choice_row(f"{source}/{r.get('id', i)}", r[text], labels, to_name(r[label]), task, source, get_lang(r), prompt, rng, k, keep_gold, suite)
            for i, r in enumerate(rows)]

In [ ]:
#| hide
fake = [{"text": "wake me at nine", "label": "alarm_set", "lang": "en"}, {"text": "pon música", "label": "play_music", "lang": "es"},
        {"text": "what's the weather", "label": "weather_query", "lang": "en"}]
r = labelled_rows(fake, "text", "label", "intent", "massive", lang="lang", k=2, keep_gold=True)
test_eq(r[1]["lang"], "es"); test_eq(r[1]["gold"]["classifications"]["intent"], "play music")
assert "play music" in r[1]["ask"]["classifications"][0]["labels"] and len(r[1]["ask"]["classifications"][0]["labels"]) == 2
test_eq(labelled_rows([{"t": "x", "y": 1}], "t", "y", "s", "src", names=["neg", "pos"], k=None)[0]["ask"]["classifications"][0]["labels"], ["neg", "pos"])

Two more sources of decisions. A typed-decision record (`state`, `questions`, `gold`, [sysone's data](02_data.ipynb)) becomes a row whose questions are GLiNER2 heads, the way [the GLiNER backend](31_gliner_backend.ipynb) asks them: a choice's criteria as labels with descriptions, a noul as yes or no, a score as its levels. And a row of Fastino's [fast-decisions](https://huggingface.co/datasets/fastino/fast-decisions) benchmark is already a text with GLiNER2 heads; its single-label heads are kept with their gold, for evaluation.

GLiNER2 finds a head's answer by the head's name, and a name that starts another (`risk` and `risk_level`) would match the wrong one, so names that are prefixes of others get a suffix.

In [ ]:
#| export
def distinct_tasks(names:list) -> dict:
    "{name: name to use}: a name that another starts with gets ' (question)' added, so GLiNER2 matches each head to its own name"
    return {n: (f"{n} (question)" if any(m != n and m.startswith(n) for m in names) else n) for n in names}

def record_rows(records, source:str="typed-decisions", keep_gold:bool=False, suite:str|None=None) -> list:
    "Rows from typed-decision records: one GLiNER2 head per question, its labels and descriptions as the GLiNER backend writes them"
    from sysone.core import load_record, render_state, as_questions
    from sysone.gliner import gliner_tasks
    out = []
    for i, rec in enumerate(records):
        rec = load_record(rec); qs = as_questions(rec["questions"]); tasks = gliner_tasks(rec["questions"])
        rename = distinct_tasks([t["task"] for t in tasks])
        cls, gold = [], {}
        for t in tasks:
            labels = t["labels"]; names = list(labels) if isinstance(labels, dict) else labels
            c = {"task": rename[t["task"]], "labels": names}
            if isinstance(labels, dict) and any(labels.values()): c["label_descriptions"] = {l: d for l, d in labels.items() if d}
            if t["instruction"].strip(): c["prompt"] = t["instruction"].strip()
            cls.append(c)
            target, k = qs[t["task"]].target((rec.get("gold") or {}).get(t["task"]))
            if target is not None: gold[c["task"]] = names[t["keys"].index(qs[t["task"]].keys[k])]
        row = {"id": f"{source}/{rec.get('id', i)}", "kind": "decision", "source": source, "lang": "en", "text": render_state(rec["state"]),
               "ask": {"classifications": cls}}
        if keep_gold and gold: row["gold"] = {"classifications": gold}
        if suite: row["suite"] = suite
        out.append(row)
    return out

def fast_decision_rows(rows:list, domain:str, suite:str="fast-decisions") -> list:
    "Rows from one fast-decisions domain: its single-label heads, with their gold"
    out = []
    for i, r in enumerate(rows):
        heads = [h for h in r["output"]["classifications"] if not h.get("multi_label")]
        rename = distinct_tasks([h["task"] for h in heads])
        out.append({"id": f"{suite}/{domain}/{i:03d}", "kind": "decision", "source": suite, "domain": domain, "lang": "en", "text": r["input"],
                    "suite": suite, "ask": {"classifications": [{"task": rename[h["task"]], "labels": list(h["labels"])} for h in heads]},
                    "gold": {"classifications": {rename[h["task"]]: h["true_label"][0] for h in heads}}})
    return out

In [ ]:
recs = [json.loads(l) for l in Path("fixtures/typed_decisions_tiny.jsonl").read_text().splitlines()]
row = record_rows(recs[:1], keep_gold=True)[0]
[(c["task"], c["labels"], c.get("prompt", "")[:40]) for c in row["ask"]["classifications"]], row["gold"]

([('action',
   ['continue', 'human_review', 'observe', 'stop'],
   'What should the observability system do '),
  ('needs_review', ['yes', 'no'], 'This trace requires human review.'),
  ('outcome',
   ['failure', 'harmful', 'partial', 'success'],
   'How did this agent run turn out?'),
  ('risk', ['0', '1', '2', '3'], "How risky was the agent's behaviour in t"),
  ('urgency',
   ['0', '1', '2', '3'],
   'How quickly does this trace need attenti')],
 {'classifications': {'action': 'human_review',
   'needs_review': 'yes',
   'outcome': 'partial',
   'risk': '0',
   'urgency': '1'}})

In [ ]:
#| hide
test_eq(distinct_tasks(["risk", "risk_level", "topic"]), {"risk": "risk (question)", "risk_level": "risk_level", "topic": "topic"})
assert all(row["gold"]["classifications"][c["task"]] in c["labels"] for c in row["ask"]["classifications"] if c["task"] in row["gold"]["classifications"])
fd = fast_decision_rows([{"input": "hi", "output": {"classifications": [{"task": "intent", "labels": ["a", "b"], "true_label": ["b"], "multi_label": False},
                                                                     {"task": "tags", "labels": ["x", "y"], "true_label": ["x", "y"], "multi_label": True}]}}], "d")
test_eq((fd[0]["ask"]["classifications"], fd[0]["gold"]), ([{"task": "intent", "labels": ["a", "b"]}], {"classifications": {"intent": "b"}}))

### Web text, with questions of its own

Labelled datasets cover a few languages and a few kinds of question. Web text covers many languages, and any question can be asked of it, the teacher supplies the answer. `web_rows` cuts texts into passages of up to `words` words and asks each one, at random:

- for **extraction**, a few entity types from `ENTITY_POOL`, sometimes relation types from `RELATION_POOL` and sometimes a record from `STRUCTURE_POOL`, the fields of something the text may describe;
- for **decisions**, one or two of the operational questions in `DECISION_POOL`: what kind of text it is, its tone, how urgent, whether it should go to a person;
- **mixed** rows ask one decision and a small extraction together, as a single GLiNER2 call can.

The questions are in English, as most schemas are, whatever the text's language.

In [ ]:
#| export
ENTITY_POOL = ["person", "organization", "company", "city", "country", "location", "date", "time", "money", "percentage", "product",
               "event", "law", "language", "nationality", "job title", "email address", "phone number", "website", "address",
               "brand", "software", "disease", "medication", "food", "vehicle", "book", "film", "song", "sports team", "university",
               "government agency", "political party", "currency", "quantity", "award", "building", "animal", "plant", "chemical"]
RELATION_POOL = ["works for", "located in", "founded by", "part of", "born in", "headquartered in", "owns", "member of", "married to",
                 "created by", "leader of", "capital of", "produces", "acquired", "studied at"]
STRUCTURE_POOL = {"company": ["name", "industry", "location", "founded"], "person": ["name", "job title", "organization", "nationality"],
                  "event": ["name", "date", "location", "participants"], "product": ["name", "brand", "price", "features"],
                  "transaction": ["buyer", "seller", "item", "amount", "date"], "job posting": ["title", "company", "location", "salary"],
                  "appointment": ["who", "with whom", "date", "time", "place"], "contact": ["name", "email", "phone", "address"]}
DECISION_POOL = [
    {"task": "text type", "labels": ["news", "opinion", "advertisement", "product page", "forum post", "encyclopedia", "instructions", "personal story", "legal or policy text", "other"],
     "prompt": "What kind of text is this?"},
    {"task": "tone", "labels": ["positive", "neutral", "negative", "mixed"], "prompt": "What is the overall tone?"},
    {"task": "urgency", "labels": ["0", "1", "2", "3", "4", "5"], "prompt": "How urgent is acting on this text, from 0 (not at all) to 5 (immediately)?"},
    {"task": "needs a human", "labels": ["yes", "no"], "prompt": "Should a person handle this rather than an automated assistant?"},
    {"task": "audience", "labels": ["general public", "experts", "customers", "employees", "children"], "prompt": "Who is this written for?"},
    {"task": "topic", "labels": ["politics", "business", "technology", "science", "health", "sports", "culture", "travel", "education", "law", "religion", "lifestyle"],
     "prompt": "What is the main topic?"},
    {"task": "contains personal data", "labels": ["yes", "no"], "prompt": "Does the text contain personal data such as names with contact details?"},
    {"task": "register", "labels": ["formal", "informal"], "prompt": "Is the language formal or informal?"},
    {"task": "request type", "labels": ["question", "complaint", "order", "cancellation", "feedback", "information", "none"],
     "prompt": "If this asks for something, what kind of request is it?"},
    {"task": "is it complete", "labels": ["complete", "cut off", "unclear"], "prompt": "Does the text read as complete?"},
]

def _words(s:str) -> int:
    "Words, counting two CJK characters as one (Chinese and Japanese put no spaces between words)"
    cjk = sum(1 for ch in s if "\u2e80" <= ch <= "\u9fff" or "\uac00" <= ch <= "\ud7af")
    return len(s.split()) + cjk // 2

def passages(text:str, words:int=120, most:int=2) -> list:
    "Up to `most` passages of up to `words` words (two CJK characters counting as one), cut at sentence ends where possible"
    sents = [x for x in re.split(r"(?<=[.!?。！？])\s*", text.strip()) if x.strip()]
    out, cur = [], []
    for s in sents:
        if _words(s) > words: s = " ".join(s.split()[:words]) if len(s.split()) > 1 else s[:2 * words]
        if cur and _words(" ".join(cur + [s])) > words:
            out.append(" ".join(cur)); cur = []
            if len(out) == most: break
        cur.append(s)
    if cur and len(out) < most: out.append(" ".join(cur))
    return [p for p in out if _words(p) >= 8]

def extraction_ask(rng:random.Random, small:bool=False) -> dict:
    "A random extraction ask: entity types, sometimes relations, sometimes a record"
    ask = {"entities": rng.sample(ENTITY_POOL, rng.randint(2, 3) if small else rng.randint(3, 7))}
    if rng.random() < (0.2 if small else 0.5): ask["relations"] = rng.sample(RELATION_POOL, rng.randint(1, 3))
    if rng.random() < (0.2 if small else 0.4):
        name = rng.choice(list(STRUCTURE_POOL)); ask["structures"] = {name: STRUCTURE_POOL[name]}
    return ask

def decision_ask(rng:random.Random, most:int=2) -> dict:
    "One or two random questions from DECISION_POOL"
    return {"classifications": [dict(c) for c in rng.sample(DECISION_POOL, rng.randint(1, most))]}

def web_rows(texts, source:str, lang:str|None=None, kinds=("extraction", "decision", "mixed"), weights=(0.45, 0.45, 0.10),
             words:int=120, seed:int=0) -> list:
    "Rows from raw texts: each passage with an extraction, decision or mixed ask, drawn with `weights`"
    rng, out = random.Random(f"{seed}:{source}:{lang}"), []
    for i, t in enumerate(texts):
        for j, p in enumerate(passages(t, words)):
            kind = rng.choices(kinds, weights)[0]
            ask = extraction_ask(rng) if kind == "extraction" else decision_ask(rng) if kind == "decision" else decision_ask(rng, 1) | extraction_ask(rng, small=True)
            out.append({"id": f"{source}/{lang}/{i}/{j}", "kind": kind, "source": source, "lang": lang, "text": p, "ask": ask})
    return out

In [ ]:
web = web_rows(["Acme Corp abrió una oficina en Madrid en 2021. Su directora, Ana Ruiz, trabajó antes en Google. " * 3,
                "Mi pedido no ha llegado y nadie responde. Quiero un reembolso hoy mismo, por favor. " * 4], "fineweb-2", "es", words=30)
[(r["kind"], r["text"][:50], r["ask"]) for r in web][:3]

[('mixed',
  'Acme Corp abrió una oficina en Madrid en 2021. Su ',
  {'classifications': [{'task': 'topic',
     'labels': ['politics',
      'business',
      'technology',
      'science',
      'health',
      'sports',
      'culture',
      'travel',
      'education',
      'law',
      'religion',
      'lifestyle'],
     'prompt': 'What is the main topic?'}],
   'entities': ['food', 'phone number']}),
 ('extraction',
  'Su directora, Ana Ruiz, trabajó antes en Google. A',
  {'entities': ['disease', 'building', 'medication'],
   'structures': {'transaction': ['buyer',
     'seller',
     'item',
     'amount',
     'date']}}),
 ('decision',
  'Mi pedido no ha llegado y nadie responde. Quiero u',
  {'classifications': [{'task': 'urgency',
     'labels': ['0', '1', '2', '3', '4', '5'],
     'prompt': 'How urgent is acting on this text, from 0 (not at all) to 5 (immediately)?'},
    {'task': 'is it complete',
     'labels': ['complete', 'cut off', 'unclear'],
     'prompt': 'Does t

In [ ]:
#| hide
assert all(len(r["text"].split()) <= 31 for r in web) and {r["kind"] for r in web} <= {"extraction", "decision", "mixed"}
mixed = [r for r in web_rows(["word " * 200] * 50, "w", "xx", kinds=("mixed",), weights=(1,), words=20)]
assert all("classifications" in r["ask"] and "entities" in r["ask"] for r in mixed)
test_eq(passages("short."), [])
cjk = passages("北京大学是中国的一所大学，位于北京市海淀区。" * 40, words=30)
assert len(cjk) == 2 and all(_words(p) <= 30 for p in cjk)

### Keeping evaluation text out of training

A training text that is also in an evaluation suite, or nearly, would inflate the scores. `near_duplicates` finds them with MinHash: each text becomes the set of its 5-word shingles, 64 hash functions keep the smallest hash of the set each, and two texts share each of those minimums with probability equal to the Jaccard similarity of their shingle sets. Texts whose signatures fall in a shared bucket (16 bands of 4) are compared, and a training text is dropped when its estimated similarity to any evaluation text reaches `threshold`, or when its normalised text equals one exactly.

In [ ]:
#| export
def shingles(text:str, n:int=5) -> set:
    "The text's n-word shingles, lowercased, spacing normalised (for a text in a script without spaces, its n-character shingles)"
    t = _norm(text); w = t.split()
    if len(w) < 3 and len(t) > 20: w = list(t.replace(" ", ""))          # Chinese, Japanese, Thai…
    return {" ".join(w[i:i + n]) for i in range(max(1, len(w) - n + 1))}

def minhash(texts:list, n:int=5, perms:int=64, seed:int=0) -> np.ndarray:
    "MinHash signatures (len(texts) × perms) of the texts' shingle sets"
    rs = np.random.RandomState(seed); P = np.uint64((1 << 61) - 1)
    a = rs.randint(1, 2**31 - 1, size=perms).astype(np.uint64); b = rs.randint(0, 2**31 - 1, size=perms).astype(np.uint64)
    out = np.empty((len(texts), perms), dtype=np.uint64)
    for i, t in enumerate(texts):
        h = np.array([int.from_bytes(hashlib.blake2b(s.encode(), digest_size=4).digest(), "little") for s in shingles(t, n)], dtype=np.uint64)
        out[i] = ((h[:, None] * a + b) % P).min(0)
    return out

def near_duplicates(texts:list, against:list, threshold:float=0.6, bands:int=16) -> set:
    "Indices of `texts` equal (normalised) or near (estimated Jaccard ≥ threshold) to some text of `against`"
    exact = {_norm(t) for t in against}
    hit = {i for i, t in enumerate(texts) if _norm(t) in exact}
    sa, st = minhash(against), minhash(texts); r = sa.shape[1] // bands
    buckets = defaultdict(list)
    for j, s in enumerate(sa):
        for k in range(bands): buckets[(k, s[k * r:(k + 1) * r].tobytes())].append(j)
    for i, s in enumerate(st):
        if i in hit: continue
        cands = {j for k in range(bands) for j in buckets.get((k, s[k * r:(k + 1) * r].tobytes()), ())}
        if any((s == sa[j]).mean() >= threshold for j in cands): hit.add(i)
    return hit

In [ ]:
evals = ["the quick brown fox jumps over the lazy dog near the river bank today", "completely different sentence about invoices and refunds for customers"]
train = ["The quick brown fox jumps over the lazy dog near the river bank today!", "the quick brown fox jumps over the lazy dog near the river bank yesterday",
         "an unrelated text on rockets, planets and the moon landing of 1969"]
near_duplicates(train, evals)

{0, 1}

In [ ]:
#| hide
test_eq(near_duplicates(train, evals), {0, 1})
test_eq(near_duplicates(["北京大学是中国的一所大学，位于北京市海淀区"], ["北京大学是中国的一所大学，位于北京市海淀区。"]), {0})

`split_rows` splits rows into training and validation by a hash of their id, so a row stays on its side however the rows are ordered or added to.

In [ ]:
#| export
def split_rows(rows:list, valid:float=0.05, seed:int=0) -> tuple:
    "Training and validation rows, by a hash of each row's id"
    h = lambda r: int(hashlib.sha1(f"{seed}:{r['id']}".encode()).hexdigest()[:8], 16) / 2**32
    return [r for r in rows if h(r) >= valid], [r for r in rows if h(r) < valid]

def parquet_rows(repo:str, config:str, split:str, n:int|None=None, seed:int=0):
    "A split of a Hub dataset from its Parquet conversion (`refs/convert/parquet`), which loads even when the dataset's own loading script no longer does"
    from datasets import load_dataset
    ds = load_dataset(repo, revision="refs/convert/parquet", data_files={split: f"{config}/{split}/*.parquet"}, split=split)
    return ds.shuffle(seed=seed).select(range(n)) if n is not None and n < len(ds) else ds

In [ ]:
#| hide
tr, va = split_rows([{"id": str(i)} for i in range(1000)], valid=0.1)
assert 60 < len(va) < 140 and len(tr) + len(va) == 1000
ids = lambda rs: sorted(r["id"] for r in rs)
test_eq(ids(split_rows([{"id": str(i)} for i in range(50)], 0.3)[1]), ids(split_rows([{"id": str(i)} for i in reversed(range(50))], 0.3)[1]))